In [2]:
import tifffile
import dask.array as da
import napari
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader

In [6]:
fn = '/mnt/DATA/slice_5375_fast_view.ome.zarr/'

# Didn't notice much difference in I/O between local and synology

In [59]:
# read the image data
reader = Reader(parse_url(fn))
# nodes may include images, labels etc
nodes = list(reader())
# first node will be the image pixel data
image_node = nodes[0]

dask_data = image_node.data

dask_data

[dask.array<from-zarr, shape=(3, 11, 41702, 58291), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 20851, 29145), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 10425, 14572), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 5212, 7286), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(3, 11, 2606, 3643), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>]

In [8]:
dask_data[0]

dask.array<from-zarr, shape=(3, 11, 41702, 58291), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>

In [9]:
viewer = napari.Viewer(title = 'local zarr speed test')
viewer.add_image(dask_data, channel_axis=0, scale = (2.0, 0.1625, 0.1625))

[<Image layer 'Image' at 0x7c1b89b5d850>,
 <Image layer 'Image [1]' at 0x7c1b483f5110>,
 <Image layer 'Image [2]' at 0x7c1a72f39b90>]

# Otsu

In [60]:
dask_data = dask_data[0]

In [62]:
dask_data

dask.array<from-zarr, shape=(3, 11, 41702, 58291), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>

In [63]:
# --- Otsu on huge uint16 dask arrays (with tqdm) ---
import numpy as np
import dask.array as da
from dask.delayed import delayed
from tqdm.auto import tqdm
from math import isfinite
import napari


C1, C2 = 1, 2
assert dask_data.dtype == np.uint16, "This helper assumes uint16; adapt bins/range otherwise."

def _histogram_block_uint16(block, n_bins=65536):
    # block is a NumPy array received inside the worker
    hist, _ = np.histogram(block, bins=n_bins, range=(0, 65536))
    return hist.astype(np.int64, copy=False)

def global_hist_uint16(darr, n_bins=65536):
    """
    Compute a global histogram over a (possibly multi-D) dask array by
    iterating blocks with tqdm (explicit progress). Returns a 1D numpy hist.
    """
    # Flatten to blocks; each entry is a delayed numpy array (a chunk)
    blocks = darr.to_delayed().ravel()
    acc = np.zeros(n_bins, dtype=np.int64)

    for blk in tqdm(blocks, desc="Computing per-chunk histograms"):
        h = delayed(_histogram_block_uint16)(blk)
        h_np = h.compute()  # compute one block at a time to update tqdm
        acc += h_np
    return acc

def otsu_from_hist(hist):
    """
    Classic Otsu threshold from a 1D histogram (bins correspond to 0..65535).
    Returns an integer threshold in [0, 65535].
    """
    hist = hist.astype(np.float64, copy=False)
    total = hist.sum()
    if total == 0:
        return 0

    # bin centers (uint16 levels)
    bins = np.arange(hist.size, dtype=np.float64)

    weight_b = np.cumsum(hist)
    weight_f = total - weight_b

    mu_b = np.cumsum(hist * bins)
    mu_f = (hist.sum()*bins.mean() - mu_b)  # not used; compute directly below

    # safer: compute global mean once, then foreground mean from complements
    mu_total = (hist * bins).sum()
    mu_b_safe = mu_b / np.maximum(weight_b, 1e-12)
    mu_f_safe = (mu_total - mu_b) / np.maximum(weight_f, 1e-12)

    between = (weight_b * weight_f) * (mu_b_safe - mu_f_safe) ** 2
    k = np.nanargmax(between)
    return int(k)

def global_otsu_uint16(darr, n_bins=65536):
    """
    Convenience: global histogram + Otsu threshold over a dask array.
    """
    hist = global_hist_uint16(darr, n_bins=n_bins)
    thr = otsu_from_hist(hist)
    return thr, hist

# --- compute global thresholds per channel (over all ZYX) ---
thr_c1, hist_c1 = global_otsu_uint16(dask_data[C1])
thr_c2, hist_c2 = global_otsu_uint16(dask_data[C2])
print(f"Otsu thresholds -> ch{C1}: {thr_c1}, ch{C2}: {thr_c2}")

# --- build lazy binary masks (same shape as original stack) ---
mask_c1 = dask_data[C1] > thr_c1
mask_c2 = dask_data[C2] > thr_c2

# Stack back to a (2, Z, Y, X) array for display; keep it lazy.
mask_stack = da.stack([mask_c1, mask_c2], axis=0)

# --- show in napari as overlays next to your existing image ---
# (re-use your scaling so masks align)
scale = (2.0, 0.1625, 0.1625)   # same as you used above

# You already created 'viewer' in your cell; if not, uncomment next line.
# viewer = napari.Viewer(title='local zarr Otsu preview')

# Add masks as an Image (binary) with additive blending, one per channel,
# or as a single layer with channel_axis. I’ll do a single layer for convenience:
viewer.add_image(
    mask_stack,
    name=['Otsu ch1','Otsu ch2'],
    channel_axis=0,
    # scale=scale,
    opacity=0.5,
    blending='additive',
    colormap=('gray','cyan'),   # visually distinct overlays
    rgb=False,
)


Computing per-chunk histograms:   0%|          | 0/102828 [00:00<?, ?it/s]

Computing per-chunk histograms:   0%|          | 0/102828 [00:00<?, ?it/s]

Otsu thresholds -> ch1: 1462, ch2: 477


/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Otsu ch1' at 0x7c186579dcd0>,
 <Image layer 'Otsu ch2' at 0x7c1927ed3490>]

In [21]:
viewer = napari.Viewer(title = 'subsection')
viewer.add_image(dask_data, channel_axis=0)
viewer.add_image(
    mask_stack,
    name=['Otsu ch1','Otsu ch2'],
    channel_axis=0,
    scale=scale,
    opacity=0.5,
    blending='additive',
    colormap=('gray','cyan'),   # visually distinct overlays
    rgb=False,
)

[<Image layer 'Otsu ch1' at 0x7c1925ba65d0>,
 <Image layer 'Otsu ch2' at 0x7c19250d9f90>]

In [22]:
for layer in viewer.layers:
    layer.scale = scale

# Manual labelling

In [26]:
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
import dask.array as da
import numpy as np, zarr, math, json
from numcodecs import Blosc
from tqdm.auto import tqdm
from skimage.filters import threshold_otsu

In [25]:
# === config ===
fn = "/mnt/DATA/slice_5375_fast_view.ome.zarr/"
C1, C2 = 1, 2                # the two latter channels
TILE = 2000                  # ROI tile size in pixels
MERGE = "or"                 # "or" or "and"
LABEL_NAME = "Otsu_C1C2_maxZ"
CHUNKS = (1024, 1024)

In [36]:
# --- open store & read multiscale image ---
url = parse_url(fn, mode="a")
reader = Reader(url)
nodes = list(reader())
img_levels = nodes[0].data              # list of (C,Z,Y,X) dask arrays
assert isinstance(img_levels, list) and img_levels, "No multiscale arrays found."

In [37]:
# full-res level (level 0)
L0 = img_levels[0]                      # dask array (C,Z,Y,X)
assert L0.ndim == 4 and L0.dtype == np.uint16

# Z max-projection per channel (lazy)
proj = L0.max(axis=1)                   # (C,Y,X)
c1 = proj[C1]                           # (Y,X)
c2 = proj[C2]

Y, X = map(int, c1.shape)

In [40]:
# ---- zarr v2/v3 compatible array creation ----
def zarr_create_compat(group, name, *, shape, chunks, dtype, overwrite=True):
    """Create an array regardless of zarr_format (2 or 3)."""
    try:
        zf = group.metadata.zarr_format  # 2 or 3
    except Exception:
        # older zarr may not expose this; treat as v2
        zf = 2

    kwargs = dict(shape=shape, chunks=chunks, dtype=dtype, overwrite=overwrite)

    if zf == 3:
        # Zarr v3: use zarr.codecs.* and `compressors=[...]`
        from zarr.codecs import Zstd
        kwargs["compressors"] = [Zstd(level=3)]
        return group.create(name, **kwargs)
    else:
        # Zarr v2: use numcodecs and `compressor=...`
        from numcodecs import Blosc
        kwargs["compressor"] = Blosc(cname="zstd", clevel=3, shuffle=Blosc.SHUFFLE)
        return group.create(name, **kwargs)


In [41]:
# --- create a working on-disk array for the base labels plane ---
store = url.store
root = zarr.open_group(store, mode="a") 

# tmp working array:
tmp_grp = root.require_group("tmp_work")
labels_base = zarr_create_compat(
    tmp_grp, "labels_base", shape=(Y, X), chunks=CHUNKS, dtype="uint8"
)

# --- per-tile Otsu with tqdm on level 0 ---
ny, nx = math.ceil(Y / TILE), math.ceil(X / TILE)
pbar = tqdm(total=ny*nx, desc="Otsu tiles (C1 & C2 @ L0 → labels)")
for iy in range(ny):
    y0, y1 = iy*TILE, min((iy+1)*TILE, Y)
    for ix in range(nx):
        x0, x1 = ix*TILE, min((ix+1)*TILE, X)

        t1 = c1[y0:y1, x0:x1].compute()
        t2 = c2[y0:y1, x0:x1].compute()

        thr1 = threshold_otsu(t1) if t1.size else 0
        thr2 = threshold_otsu(t2) if t2.size else 0
        m1, m2 = t1 > thr1, t2 > thr2
        m = (m1 & m2) if MERGE == "and" else (m1 | m2)

        labels_base[y0:y1, x0:x1] = m.astype(np.uint8, copy=False)
        pbar.update(1)
pbar.close()

# --- build labels pyramid to match the image multiscales ---
def downscale_max(arr):
    # downscale by 2 with max pooling (preserves labels)
    y, x = arr.shape
    y2, x2 = y // 2, x // 2
    return arr[:y2*2, :x2*2].reshape(y2, 2, x2, 2).max(axis=(1,3))

# labels pyramid level 0:
labels_root = root.require_group("labels").require_group("0")
L0_lbl = zarr_create_compat(
    labels_root, "0", shape=(Y, X), chunks=CHUNKS, dtype="uint8"
)

L0_lbl[...] = labels_base[...]
# lvl_paths.append("0")


/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/zarr/codecs/numcodecs/_codecs.py:139: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


Otsu tiles (C1 & C2 @ L0 → labels):   0%|          | 0/630 [00:00<?, ?it/s]

NameError: name 'lvl_paths' is not defined

In [ ]:
# --- config ---
fn = "/mnt/DATA/slice_5375_fast_view.ome.zarr/"
C1, C2 = 1, 2              # channels to threshold
TILE = 2000                # tile size for Otsu
MERGE = "or"               # "or" or "and"
LABEL_NAME = "Otsu_C1C2_maxZ"
CHUNKS = (1024, 1024)

# --- imports ---
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
import dask.array as da
import numpy as np, zarr, math
from tqdm.auto import tqdm
from skimage.filters import threshold_otsu

# zarr v2/v3 compatible array creator
def zarr_create_compat(group, name, *, shape, chunks, dtype, overwrite=True):
    try:
        zf = group.metadata.zarr_format  # 2 or 3
    except Exception:
        zf = 2
    kwargs = dict(shape=shape, chunks=chunks, dtype=dtype, overwrite=overwrite)
    if zf == 3:
        from zarr.codecs import Zstd
        kwargs["compressors"] = [Zstd(level=3)]
    else:
        from numcodecs import Blosc
        kwargs["compressor"] = Blosc(cname="zstd", clevel=3, shuffle=Blosc.SHUFFLE)
    return group.create(name, **kwargs)

def downscale_max(arr):
    # downscale by 2 with max pooling (preserves labels)
    y, x = arr.shape
    y2, x2 = y // 2, x // 2
    return arr[:y2*2, :x2*2].reshape(y2, 2, x2, 2).max(axis=(1,3))

def _get_img_multiscales_group(root):
    # try common places: root attrs, or group "0"
    ms = root.attrs.get("multiscales")
    if ms:
        return root, ms
    if "0" in root and isinstance(root["0"], zarr.Group):
        g0 = root["0"]
        ms0 = g0.attrs.get("multiscales")
        if ms0:
            return g0, ms0
    return None, None

# --- open store writable & read image pyramid ---
url = parse_url(fn, mode="a")        # <-- writable
reader = Reader(url)
nodes = list(reader())
img_levels = nodes[0].data           # list of dask arrays (C,Z,Y,X)
assert isinstance(img_levels, list) and len(img_levels) > 0
L0 = img_levels[0]
assert L0.ndim == 4

# --- max project Z, pick channels, shape ---
proj = L0.max(axis=1)                # (C,Y,X), lazy
c1, c2 = proj[C1], proj[C2]
Y, X = map(int, c1.shape)

# --- working tmp array to stream-write base labels ---
root = zarr.open_group(url.store, mode="a")
tmp_grp = root.require_group("tmp_work")
labels_base = zarr_create_compat(tmp_grp, "labels_base", shape=(Y, X), chunks=CHUNKS, dtype="uint8")

# --- per-tile Otsu with tqdm ---
ny, nx = math.ceil(Y/TILE), math.ceil(X/TILE)
pbar = tqdm(total=ny*nx, desc="Otsu tiles (C1 & C2 @ L0 → labels)")
for iy in range(ny):
    y0, y1 = iy*TILE, min((iy+1)*TILE, Y)
    for ix in range(nx):
        x0, x1 = ix*TILE, min((ix+1)*TILE, X)
        t1 = c1[y0:y1, x0:x1].compute()
        t2 = c2[y0:y1, x0:x1].compute()
        thr1 = threshold_otsu(t1) if t1.size else 0
        thr2 = threshold_otsu(t2) if t2.size else 0
        m1, m2 = (t1 > thr1), (t2 > thr2)
        m = (m1 & m2) if MERGE == "and" else (m1 | m2)
        labels_base[y0:y1, x0:x1] = m.astype(np.uint8, copy=False)
        pbar.update(1)
pbar.close()

# --- build labels pyramid to match image multiscales ---
labels_root = root.require_group("labels").require_group("0")
lvl_paths = []  # <-- define before using

# level 0
L0_lbl = zarr_create_compat(labels_root, "0", shape=(Y, X), chunks=CHUNKS, dtype="uint8")
L0_lbl[...] = labels_base[...]
lvl_paths.append("0")

# as many levels as image has
n_levels = len(img_levels)
prev = L0_lbl
for i in range(1, n_levels):
    y, x = prev.shape
    y2, x2 = (y + 1)//2, (x + 1)//2
    lvl = zarr_create_compat(labels_root, str(i), shape=(y2, x2), chunks=CHUNKS, dtype="uint8")
    stripe = 4096
    for ys in tqdm(range(0, y, stripe), desc=f"Downscale labels L{i-1}→L{i}", leave=False):
        ye = min(ys + stripe, y)
        slab = prev[ys:ye, :]        # NumPy
        ds = downscale_max(slab)
        lvl[ys//2: ys//2 + ds.shape[0], 0: ds.shape[1]] = ds
    prev = lvl
    lvl_paths.append(str(i))

# cleanup tmp
try:
    del root["tmp_work"]
except KeyError:
    pass



In [48]:
# --- attach NGFF labels metadata (copy YX transforms) ---

img_group, ms = _get_img_multiscales_group(root)

axes_yx = [{"name": "y", "type": "space"}, {"name": "x", "type": "space"}]

# If you want physical units, set base pixel size here (e.g., micrometers/px)
BASE_PX = None  # e.g. (0.1625, 0.1625)  # set to None to keep unitless

lvl_paths = [k for k in labels_root.array_keys()]          # e.g. ["0","1","2",...]
lvl_paths = sorted(lvl_paths, key=lambda s: int(s))        # numeric order

if ms:
    # Copy per-level transforms from the image (C,Z,Y,X) => keep YX
    img_dsets = ms[0]["datasets"]
    # If image has fewer levels than labels (unlikely), clip to min
    n_copy = min(len(img_dsets), len(lvl_paths))
    ct_yx = []
    for i in range(n_copy):
        cts = img_dsets[i].get("coordinateTransformations", [])
        out = []
        for t in cts:
            if t.get("type") == "scale":
                sc = t["scale"]
                out.append({"type": "scale", "scale": [sc[-2], sc[-1]]})
            elif t.get("type") == "translation":
                tr = t["translation"]
                out.append({"type": "translation", "translation": [tr[-2], tr[-1]]})
        if not out:
            out = [{"type": "scale", "scale": [1.0, 1.0]}]
        ct_yx.append(out)
else:
    # Synthesize transforms from level shapes (assume L0 is reference)
    # Use image level shapes if available; else use labels shapes.
    Y0, X0 = int(L0.shape[-2]), int(L0.shape[-1])
    ct_yx = []
    for p in lvl_paths:
        Yi, Xi = labels_root[p].shape
        sy = Y0 / Yi
        sx = X0 / Xi
        if BASE_PX is not None:
            sy *= BASE_PX[0]
            sx *= BASE_PX[1]
        ct_yx.append([{"type": "scale", "scale": [sy, sx]}])

labels_root.attrs["multiscales"] = [{
    "version": (ms[0].get("version", "0.4") if ms else "0.4"),
    "name": LABEL_NAME,
    "axes": ( [a for a in (ms[0]["axes"] if ms else axes_yx) if a["name"] in ("y","x")]
              if ms else axes_yx ),
    "datasets": [
        {"path": lvl_paths[i], "coordinateTransformations": ct_yx[i]}
        for i in range(len(lvl_paths))
    ],
}]

labels_root.attrs["image-label"] = {
    "version": "0.4",
    "colors": [{"label": 1, "rgba": [0, 255, 255, 255]}],
    "properties": [{"label": 1, "name": "Otsu(C1|C2)", "visible": True}],
    "source": {"image": "0"},
}

# register at root (or image group if that’s where image lives—root is fine)
reg = root.attrs.get("labels", [])
reg = [e for e in reg if e.get("path") != "labels/0"]
reg.append({"path": "labels/0", "type": "label"})
root.attrs["labels"] = reg

print("✓ Attached NGFF labels metadata (copied or synthesized transforms).")

✓ Attached NGFF labels metadata (copied or synthesized transforms).


In [49]:
import napari
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader

fn = "/mnt/DATA/slice_5375_fast_view.ome.zarr/"

# load both image and labels using the OME-Zarr reader
url = parse_url(fn)
reader = Reader(url)
nodes = list(reader())

viewer = napari.Viewer()
for node in nodes:
    data = node.data
    if isinstance(data, list):            # multiscale image
        viewer.add_image(data, name="Image", channel_axis=0, blending="additive", opacity=0.8)
    else:                                 # labels
        viewer.add_labels(data, name=node.metadata.get("name", "Labels"), opacity=0.5)

napari.run()


AttributeError: 'list' object has no attribute 'shape'

In [57]:
labels_base

<Array file:///mnt/DATA/slice_5375_fast_view.ome.zarr/tmp_work/labels_base shape=(41702, 58291) dtype=uint8>

In [58]:
viewer.add_image(
    labels_base,    # name=['Otsu ch1','Otsu ch2'],
    # channel_axis=0,
    # scale=scale,
    opacity=0.5,
    blending='additive',
    # colormap=('gray','cyan'),   # visually distinct overlays
    # rgb=False,
)


/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Image layer 'labels_base' at 0x7c191e1ab6d0>

### Increasing I/O speed

In [48]:
dask_image_ROI = dask_image[:,:,0:10000, 0:10000]

In [49]:
dask_image_ROI

dask.array<getitem, shape=(3, 11, 10000, 10000), dtype=uint16, chunksize=(1, 1, 512, 512), chunktype=numpy.ndarray>

In [ ]:
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image, add_metadata

# your dask image: (Z, Y, X, C)
dimg_czyx = dimg.transpose(3, 0, 1, 2).rechunk((1, 1, 1024, 1024))

from zarr.codecs import Blosc  # Zarr v3 (NGFF v0.5)
compressors = (Blosc(cname="lz4", clevel=1, shuffle=Blosc.SHUFFLE),)

path = "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/zarr/slice_5375_fast_view.ome.zarr"

store = parse_url(path, mode="w").store
root = zarr.group(store=store)

with ProgressBar():  # optional live progress
    write_image(
        image=dask_image,
        group=root,
        axes="czyx",
        storage_options=dict(chunks=(1, 1, 512, 512),)#) compressors=compressors),
    )

# optional labels so you can toggle channels quickly
add_metadata(root, {"omero": {"channels": [
    {"label": "CF405"}, {"label": "CF488"}, {"label": "CF561"}
]}})


[#####################                   ] | 53% Completed | 2hr 59ms

IOStream.flush timed out


[#######################                 ] | 58% Completed | 3hr 16m

# Todo:

1. ~Increase speed of loading by reducing chunk size~
2. Create rapid label layer
3. Refine I/O process

### convert to a pyramidal OME-TIFF for napari using bftools in command line
(godspee) dayn@9GRLVQ3:/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/tif$ bfconvert mtb_slice_5375_s0.ome.tif mtb_slice_5375_pyr.ome.tiff -bigtiff -pyramid-scale 2 -pyramid-resolutions 5 -tilex 512 -tiley 512 -compression LZW



### View in napari

In [6]:
import napari

In [7]:
viewer = napari.Viewer()
viewer.add_image(image)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291, 3) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Image layer 'image' at 0x7c21dc5e6750>

In [8]:
image.shape

(11, 41702, 58291, 3)

In [9]:
viewer.add_image(image, channel_axis=-1)

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


[<Image layer 'Image' at 0x7c222aa83b50>,
 <Image layer 'Image [1]' at 0x7c21dcf02cd0>,
 <Image layer 'Image [2]' at 0x7c21dc285bd0>]

/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (11, 41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 3D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/godspee/lib/python3.11/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (41702, 58291) exceeds GL_MAX_TEXTU

# My solution

In [24]:
image.shape

(11, 41702, 58291, 3)

# Chatgpt guff below

In [10]:
import numpy as np
import zarr
from openslide import OpenSlide


In [ ]:
OpenSlide.

In [ ]:
#!/usr/bin/env python3
# Minimal TIFF (z,y,x,c) -> pyramidal OME-Zarr (multiscales)

from pathlib import Path
import numpy as np
import tifffile as tiff
import dask.array as da
from dask.array import coarsen
import zarr
from numcodecs import Blosc
from ome_zarr.writer import write_multiscales_metadata
from tqdm.auto import tqdm

# --------- inputs / knobs ----------
tif_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/tif/mtb_slice_5375_s0.ome.tif")
out_zarr = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/zarr/slice_5375.zarr")


# chunking (z,y,x,c)
CHUNK = (1, 2048, 2048, 3)
COMP  = Blosc(cname="zstd", clevel=5, shuffle=Blosc.BITSHUFFLE)

# stop pyramid when both y & x <= this
TARGET_MIN_YX = 1024

# from your metadata
px_um = dict(z=2.0, y=0.1625, x=0.1625)
channel_labels = ["CF405", "CF488", "CF561"]
# ------------------------------------

# open lazily
asz = tiff.imread(tif_path, aszarr=True)
src = da.from_zarr(asz)  # expects (z,y,x,c)
assert src.ndim == 4, f"expected (z,y,x,c), got {src.shape}"
src = src.rechunk(CHUNK)

# helper: pad Y/X up to next multiple of 'factor'
def pad_even_yx(a, factor=2):
    pad = [(0,0)] * a.ndim
    for ax in (1, 2):  # y, x
        rem = a.shape[ax] % factor
        if rem:
            pad[ax] = (0, factor - rem)
    # edge padding duplicates edge pixels (cheap + safe for overviews)
    return da.pad(a, pad, mode="edge")

# build pyramid (2x YX) using safe coarsen
levels = [src]
scales = [[px_um["z"], px_um["y"], px_um["x"], 1.0]]

arr = src
while max(arr.shape[1:3]) > TARGET_MIN_YX:
    arr_even = pad_even_yx(arr, factor=2)
    arr = coarsen(np.mean, arr_even, {1: 2, 2: 2}, trim_excess=False)
    arr = arr.rechunk(CHUNK)
    levels.append(arr)
    i = len(levels) - 1
    scales.append([px_um["z"], px_um["y"] * 2**i, px_um["x"] * 2**i, 1.0])



In [17]:
src

dask.array<rechunk-merge, shape=(11, 3, 41702, 58291), dtype=uint16, chunksize=(1, 3, 2048, 3), chunktype=numpy.ndarray>

In [19]:
# ---- Zarr v2/v3 compressor shim ----
try:
    # Zarr v3 path
    from zarr.codecs import Blosc as ZarrBlosc
    CREATE_KW = {
        "compressors": (ZarrBlosc(cname="zstd", clevel=5, shuffle=ZarrBlosc.BITSHUFFLE),),
    }
except Exception:
    # Fallback for Zarr v2
    from numcodecs import Blosc as NumBlosc
    print('here')
    CREATE_KW = {
        "compressor": NumBlosc(cname="zstd", clevel=5, shuffle=NumBlosc.BITSHUFFLE),
    }


here


In [21]:
# zarr group
root = zarr.open_group(str(out_zarr), mode="w")


# chunk offset helper (for tqdm block writes)
def chunk_offsets(chunks_tuple):
    offs = [0]
    s = 0
    for ch in chunks_tuple[:-1]:
        s += ch
        offs.append(s)
    return offs

# --- stream-write each level with tqdm (Zarr v3-safe) ---
for i, arr in enumerate(levels):
    create_fn = getattr(root, "create_array", None) or root.create_dataset  # v3 then v2
    ds = create_fn(
        str(i),
        shape=arr.shape,
        chunks=CHUNK,
        dtype=arr.dtype,
        overwrite=True,
        # **CREATE_KW,  # <- key change here
    )

    nz, ny, nx, nc = arr.numblocks
    total_blocks = nz * ny * nx * nc

    def _offs(ch):
        o, out = 0, [0]
        for s in ch[:-1]:
            o += s
            out.append(o)
        return out

    z_off = _offs(arr.chunks[0])
    y_off = _offs(arr.chunks[1])
    x_off = _offs(arr.chunks[2])
    c_off = _offs(arr.chunks[3])

    from tqdm.auto import tqdm
    with tqdm(total=total_blocks, desc=f"Writing level {i} {arr.shape}") as pbar:
        for bz in range(nz):
            for by in range(ny):
                for bx in range(nx):
                    for bc in range(nc):
                        block = arr.blocks[bz, by, bx, bc].compute()
                        z0, y0, x0, c0 = z_off[bz], y_off[by], x_off[bx], c_off[bc]
                        z1, y1, x1, c1 = z0 + block.shape[0], y0 + block.shape[1], x0 + block.shape[2], c0 + block.shape[3]
                        ds[z0:z1, y0:y1, x0:x1, c0:c1] = block
                        pbar.update(1)


# multiscales metadata
axes = [
    {"name": "z", "type": "space", "unit": "micrometer"},
    {"name": "y", "type": "space", "unit": "micrometer"},
    {"name": "x", "type": "space", "unit": "micrometer"},
    {"name": "c", "type": "channel"},
]
datasets = [{"path": str(i)} for i in range(len(levels))]
coord = [
    [
        {"type": "scale", "scale": s},
        {"type": "translation", "translation": [0, 0, 0, 0]},
    ]
    for s in scales
]
write_multiscales_metadata(root, datasets=datasets, axes=axes, coordinateTransformations=coord)

# optional: simple OMERO block with channel labels (keeps viewers happy)
root.attrs["omero"] = {
    "name": out_zarr.stem,
    "channels": [{"label": lbl} for lbl in channel_labels],
}

print(f"Done: {out_zarr} with {len(levels)} levels")


Writing level 0 (11, 3, 41702, 58291):   0%|          | 0/4488561 [00:00<?, ?it/s]

KeyboardInterrupt: 